# Friedkin-Johnsen Rollout Optimization

- Goal: learn one global function from stacked trajectories that maps current run state to dynamic FJ parameters, then evaluate capped full rollouts per run.
- Core dynamics at each step:

$$z(t+1)=\lambda b_t + (1-\lambda)W_t z(t)$$

- Objects:
- $b_t\in\mathbb{R}^N$ is generated from current state and run graph.
- $W_t\in\mathbb{R}^{N\times N}$ is generated from current state and run graph.
- $\lambda\in(0,1)$ is selected globally by grid search using all runs.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
from matplotlib.lines import Line2D

ROOT = Path('../..').resolve()
CLEAN = ROOT / 'cleaned_data'

INIT_ROOT = Path('../..').joinpath('..').resolve()
PAIR_DIR = INIT_ROOT / 'tests' / 'single_shot_tests' / 'data' / 'stance_converted'

PARAMS = {
    'stance_clip': (-1.0, 1.0),
    'lambda_grid': np.linspace(0.05, 0.95, 19),
    'rollout_horizon_cap': 20,
    'validation_horizon': 8,
}

PREDICTOR_PARAMS = {
    'basis_grid': ['linear', 'linear_abs', 'quadratic', 'quadratic_abs'],
    'ridge_alphas': [0.0, 0.1, 1.0, 10.0],
    'noise_bins': 6,
    'rng_seed': 0,
}

USE_PREDICTED_INIT_WHEN_NO_SLICE0_SELF_POST = True

RUN_DIRS = sorted([p for p in CLEAN.iterdir() if p.is_dir()])
print('Project root:', ROOT)
print('Cleaned data:', CLEAN)
print('Runs found:', len(RUN_DIRS))
print('Rollout horizon cap:', PARAMS['rollout_horizon_cap'])
print('Pair dir:', PAIR_DIR)
for r in RUN_DIRS[:26]:
    print(' ', r.name)

## Data Construction

- Per-run, per-agent stance trajectory is represented as $x_i^{(r)}(t)$.
- Initialization uses run-derived signals and global fallback priors from observed runs.
- Each run is mapped onto a shared global agent index.
- Neighbor sets are built per run from that run's graph: $\mathcal{N}^{(r)}_i$.

In [ ]:
from ..data_prep import load_run_data, build_global_init_map, build_run_trajectory, build_neighbors_index, _numeric_agent_key
RUN_DATA = {r.name: load_run_data(r) for r in RUN_DIRS}
GLOBAL_AGENT_IDS = sorted({a for d in RUN_DATA.values() for a in d['agent_ids']}, key=_numeric_agent_key)
N = len(GLOBAL_AGENT_IDS)

GLOBAL_INIT_BY_AGENT = build_global_init_map(RUN_DATA, GLOBAL_AGENT_IDS)
RUN_TRAJ_AND_MASK = {
    rn: build_run_trajectory(d, GLOBAL_AGENT_IDS, target_agent_fraction=0.7, return_post_mask=True, constrain_messages=150)
    for rn, d in RUN_DATA.items()
}
RUN_TRAJ = {rn: tm[0] for rn, tm in RUN_TRAJ_AND_MASK.items()}
RUN_POST_MASK = {rn: tm[1] for rn, tm in RUN_TRAJ_AND_MASK.items()}
RUN_NEIGHBORS = {rn: build_neighbors_index(d, GLOBAL_AGENT_IDS) for rn, d in RUN_DATA.items()}

print('Global agent count:', N)
print('Global init priors available:', len(GLOBAL_INIT_BY_AGENT))
for rn in sorted(RUN_TRAJ.keys()):
    tr = RUN_TRAJ[rn]
    print(f"{rn}: trajectory shape={tr.shape}, horizon={tr.shape[0] - 1}")

In [ ]:
# RUN_TRAJ posting audit: use post mask to separate true posts from feed-forward values.
print('RUN_TRAJ post-rate audit (posts / N per time slice, from RUN_POST_MASK)')
print('N =', N)
all_post_fracs = []
for rn in sorted(RUN_TRAJ.keys()):
    tr = np.asarray(RUN_TRAJ[rn], dtype=float)
    pm = np.asarray(RUN_POST_MASK[rn], dtype=bool)
    per_slice_post_frac = pm.mean(axis=1)
    all_post_fracs.extend(per_slice_post_frac.tolist())
    print(f"{rn}: post_mean={np.mean(per_slice_post_frac):.3f}, post_min={np.min(per_slice_post_frac):.3f}, post_max={np.max(per_slice_post_frac):.3f}, slices={tr.shape[0]}")

if all_post_fracs:
    print('\nGlobal RUN_TRAJ post-rate summary:')
    print(f"mean={np.mean(all_post_fracs):.3f}, median={np.median(all_post_fracs):.3f}, min={np.min(all_post_fracs):.3f}, max={np.max(all_post_fracs):.3f}, total_slices={len(all_post_fracs)}")


## Parameter Estimation

- Candidate set: $\lambda\in(0,1)$ from the configured grid.
- For each fixed $\lambda$, fit one global parameter function using stacked transitions from all runs.
- Inputs per transition: current opinions and run-neighbor graph structure.
- Outputs per transition: generated $W_t$ and $b_t$ used in the next-state equation.
- Transition objective:

$$\operatorname{MSE}=\frac{1}{M}\sum_{m=1}^{M}\|x_{m+1}-\hat{x}_{m+1}\|_2^2$$

- Constraints enforced in generation:
- neighbor-only support in each run
- $W_t$ row-wise nonnegative softmax normalization
- $b_t\in[-1,1]$ after clipping

Specifically weightings are learned from $θ$ in the form that:
- $θ[0]$ refers to an offset
- $θ[1]$ refers to a reward coefficeint for homophliy similarity
- $θ[2]$ refers to the term for homophily with respect to neighbor opinions

Bias is computed per agent as a combination of these features. 
- $b_i(t)$ = $c_0$ + $c_1 * x_it(t)$ + $c_2 * μ_i(t)$ where $μ_i(t)$ is the mean of neighboring opinions

$W_{ij}$ is computed as the row softmax of each $score_{ij}$ = $a_0$ + $a_1x_j(t)$ - $a_2$|$x_i(t)$ - $x_j(t)$|

### Agent-Specific Strength Variant ($W=\mathrm{diag}(\alpha)\,\bar A$)

A more flexible linear baseline keeps the same row-stochastic mixing matrix $\bar A^{(r)}$ (derived from run $r$), but allows each agent to have its own **neighbor influence strength** $\alpha_i\in[0,1]$.

Define $\alpha\in[0,1]^N$ and

$$W^{(r)} = \mathrm{diag}(\alpha)\,\bar A^{(r)}.$$

Each row of $W$ sums to at most $\alpha_i$, so the operator remains stable when $\alpha_i\le 1$. The update is then
$$z^{(r)}(t+1)=\lambda\,b^{(r)}+(1-\lambda)\,W^{(r)}z^{(r)}(t),\quad b^{(r)}=z^{(r)}(0).$$

We fit $(\lambda,\alpha)$ from one-step transitions pooled across runs, using a constrained least-squares fit for each agent's $\alpha_i$ (with clipping to $[0,1]$).

In [ ]:
from scipy.optimize import minimize

def build_dataset_from_run(run):
    X = []
    Y = []
    for t in range(len(run) - 1):
        X.append(run[t])
        Y.append(run[t + 1])
    X = np.array(X, dtype=float)
    Y = np.array(Y, dtype=float)
    return X, Y


def softmax_stable(v):
    z = np.asarray(v, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    s = np.sum(e)
    if s <= 1e-12:
        return np.ones_like(z) / float(len(z))
    return e / s


def generate_W_b_from_state(x, neighbors, theta_w, theta_b):
    x = np.asarray(x, dtype=float)
    N = x.shape[0]
    W = np.zeros((N, N), dtype=float)
    b = np.zeros((N,), dtype=float)

    a0, a1, a2 = theta_w
    c0, c1, c2 = theta_b

    for i in range(N):
        # 1. Get neighbors
        ns = neighbors.get(i, [i])
        if not ns: ns = [i]

        x_i = float(x[i])
        x_ns = x[ns]
        
        # 2. Linear Weight Calculation (No Softmax)
        # Calculate raw scores for neighbors
        scores = a0 + a1 * x_ns - a2 * np.abs(x_i - x_ns)
        
        score_sum = np.sum(scores)
        if abs(score_sum) > 1e-9:
            w_row = scores / score_sum
        else:
            w_row = np.ones_like(scores) / len(scores)

        for k, j in enumerate(ns):
            W[i, j] = float(w_row[k])

        neigh_mean = float(np.mean(x_ns))
        b[i] = c0 + c1 * x_i + c2 * neigh_mean

    return W, b


def predict_next_state_from_function(x, neighbors, lam, theta_w, theta_b):
    W, b = generate_W_b_from_state(x, neighbors, theta_w, theta_b)
    pred = lam * b + (1.0 - lam) * (W @ x)
    return pred, W, b


def fit_global_function_at_lambda(run_records, lam):
    lam = float(lam)
    if not (0.0 < lam < 1.0):
        raise ValueError('lambda must be in (0, 1).')

    def objective(params):
        theta_w = params[:3]
        theta_b = params[3:]
        sq_err = 0.0
        denom = 0

        for rec in run_records:
            run = rec['trajectory']
            neighbors = rec['neighbors']
            for t in range(run.shape[0] - 1):
                x_t = run[t]
                y_t1 = run[t + 1]
                pred, _, _ = predict_next_state_from_function(x_t, neighbors, lam, theta_w, theta_b)
                sq_err += float(np.sum((y_t1 - pred) ** 2))
                denom += int(y_t1.size)

        if denom == 0:
            return np.inf
        return sq_err / float(denom)

    init = np.array([0.0, 1.0, 0.5, 0.0, 1.0, 0.5], dtype=float)
    opt = minimize(objective, x0=init, method='L-BFGS-B')
    theta_w = opt.x[:3]
    theta_b = opt.x[3:]
    loss = float(objective(opt.x))

    spectral_radius_max = 0.0
    eigenvalues_by_transition = []
    all_eigenvalues = []

    for rec in run_records:
        run_name = rec.get('run_name', 'unknown_run')
        run = rec['trajectory']
        neighbors = rec['neighbors']
        for t in range(run.shape[0] - 1):
            _, W_t, _ = predict_next_state_from_function(run[t], neighbors, lam, theta_w, theta_b)
            eigvals_t = np.linalg.eigvals((1.0 - lam) * W_t)
            eigvals_list = [complex(ev) for ev in np.asarray(eigvals_t).ravel()]
            spectral_radius_t = float(np.max(np.abs(eigvals_t)))

            spectral_radius_max = max(spectral_radius_max, spectral_radius_t)
            all_eigenvalues.extend(eigvals_list)
            eigenvalues_by_transition.append(
                {
                    'run': run_name,
                    'time_index': int(t),
                    'eigenvalues': eigvals_list,
                    'spectral_radius': spectral_radius_t,
                }
            )

    return {
        'lambda': lam,
        'theta_w': np.asarray(theta_w, dtype=float),
        'theta_b': np.asarray(theta_b, dtype=float),
        'loss': loss,
        'spectral_radius': float(spectral_radius_max),
        'all_eigenvalues': np.asarray(all_eigenvalues, dtype=complex),
        'eigenvalues_by_transition': eigenvalues_by_transition,
    }


def fit_global_function_model(run_records, lambda_grid):
    per_lambda_models = []
    for lam in lambda_grid:
        m = fit_global_function_at_lambda(run_records, lam)
        per_lambda_models.append(m)

    diag_rows = [
        {
            'lambda': m['lambda'],
            'loss': m['loss'],
            'spectral_radius': m['spectral_radius'],
        }
        for m in per_lambda_models
    ]
    lambda_diagnostics = pd.DataFrame(diag_rows).sort_values('lambda').reset_index(drop=True)

    admissible = [m for m in per_lambda_models if np.isfinite(m['spectral_radius']) and m['spectral_radius'] < 1.0]
    if not admissible:
        raise RuntimeError('No admissible lambda found in the provided grid.')

    best = min(admissible, key=lambda m: m['loss'])
    model = dict(best)
    model['lambda_diagnostics'] = lambda_diagnostics
    model['per_lambda_models'] = per_lambda_models
    return model


def rollout_predict_dynamic(z0, neighbors, model, T):
    lam = float(model['lambda'])
    theta_w = np.asarray(model['theta_w'], dtype=float)
    theta_b = np.asarray(model['theta_b'], dtype=float)

    z = np.array(z0, dtype=float).copy()
    trajectory = [z.copy()]
    Ws = []
    bs = []
    for _ in range(T):
        z_next, W_t, b_t = predict_next_state_from_function(z, neighbors, lam, theta_w, theta_b)
        Ws.append(W_t)
        bs.append(b_t)
        z = z_next
        trajectory.append(z.copy())

    return np.vstack(trajectory), Ws, bs


def multi_step_validation_error_dynamic(run, neighbors, model, K=None):
    if K is None:
        K = len(run)
    K = int(min(K, len(run)))
    if K <= 1:
        return np.nan

    pred, _, _ = rollout_predict_dynamic(run[0], neighbors=neighbors, model=model, T=K - 1)
    return float(np.mean((pred[1:K] - run[1:K]) ** 2))


def run_diagnostics_table(run_name, run, neighbors, model, validation_horizon=None):
    rollout, _, _ = rollout_predict_dynamic(run[0], neighbors=neighbors, model=model, T=len(run) - 1)
    mse_rollout = float(np.mean((run[1:] - rollout[1:]) ** 2))
    mse_multi = multi_step_validation_error_dynamic(run, neighbors=neighbors, model=model, K=validation_horizon)

    return {
        'run': run_name,
        'lambda': float(model['lambda']),
        'loss_train_global': float(model['loss']),
        'mse_rollout_full': mse_rollout,
        'mse_multistep_mandatory': float(mse_multi),
        'spectral_radius_max': float(model['spectral_radius']),
    }

In [ ]:
from scipy.optimize import minimize
def adjacency_to_row_stochastic(neighbors, n):
    """Build row-stochastic adjacency matrix from a neighbor dict.

    neighbors: dict[str, dict[str, float]] or dict[str, list[str]]
      - If dict-of-dict: weights are used (must be >=0).
      - If dict-of-list: treated as unweighted edges (weight=1).
    Returns Abar with shape (n,n) where rows sum to 1 when out-degree>0.
    """
    agent_ids = range(n)
    idx = {a: i for i, a in enumerate(agent_ids)}
    A = np.zeros((n, n), dtype=float)
    for src, outs in (neighbors or {}).items():
        if src not in idx:
            continue
        i = idx[src]
        if isinstance(outs, dict):
            for dst, w in outs.items():
                if dst in idx and w is not None:
                    A[i, idx[dst]] += float(w)
        else:
            for dst in outs:
                if dst in idx:
                    A[i, idx[dst]] += 1.0
    row_sum = A.sum(axis=1, keepdims=True)
    Abar = np.divide(A, row_sum, out=np.zeros_like(A), where=row_sum > 0)
    return Abar


def bias_parametric(z_t, mu_t, c):
    """Per-agent bias vector b(t) = c0 + c1*z_t + c2*mu_t.

    c: array-like shape (3,)
    """
    c = np.asarray(c, dtype=float).ravel()
    if c.shape[0] != 3:
        raise ValueError(f"c must have shape (3,), got {c.shape}")
    return c[0] + c[1] * z_t + c[2] * mu_t


def rollout_predict_agent_alpha_param_bias(z0, *, Abar, lam, alpha_vec, T, c):
    """Rollout for z(t+1)=lam*b(t)+(1-lam)*W*z(t), with W=diag(alpha)*Abar and b(t)=c0+c1*z(t)+c2*mu(t)."""
    z0 = np.asarray(z0, dtype=float).ravel()
    n = z0.shape[0]
    T = int(T)
    alpha_vec = np.asarray(alpha_vec, dtype=float).ravel()
    if alpha_vec.shape[0] != n:
        raise ValueError(f"alpha_vec must have shape ({n},), got {alpha_vec.shape}")
    W = np.clip(alpha_vec, 0.0, 1.0)[:, None] * np.asarray(Abar, dtype=float)
    out = np.zeros((T + 1, n), dtype=float)
    out[0] = z0
    for t in range(T):
        mu = Abar @ out[t]
        b = bias_parametric(out[t], mu, c)
        out[t + 1] = float(lam) * b + (1.0 - float(lam)) * (W @ out[t])
    return out

def predict_state(z, Abar, alpha, c, lam):
    """Simplified forward step matching the original logic."""
    mu = Abar @ z
    # Bias term: c0 + c1*z + c2*mu
    b = c[0] + c[1] * z + c[2] * mu
    W = alpha[:, None] * Abar
    return lam * b + (1.0 - lam) * (W @ z)

def fit_adj_function_at_lambda(run_records, lam):
    """
    Fits alpha_vec and bias coefficients c for a fixed lambda.
    Formula: z(t+1) = lam * b(t) + (1 - lam) * (W @ z(t))
    where W = diag(alpha) @ Abar and b(t) = c0 + c1*z(t) + c2*mu(t)
    """
    lam = float(lam)
    # Get N from the first trajectory to define parameter vector size
    n_states = run_records[0]['trajectory'].shape[1]

    # Pre-compute row-stochastic matrices once to save overhead
    run_cache = []
    for rec in run_records:
        run_name = rec.get('run_name', 'unknown_run')
        Abar = adjacency_to_row_stochastic(rec['neighbors'], n_states)
        run_cache.append((run_name, np.asarray(rec['trajectory']), Abar))

    def objective(params):
        alpha_vec = params[:n_states]
        c_vec = params[n_states:] # [c0, c1, c2]

        total_sq_err = 0.0
        count = 0

        for _, x, Abar in run_cache:
            z_t = x[:-1]      # Shape (T-1, N)
            target = x[1:]    # Shape (T-1, N)

            mu_t = z_t @ Abar.T

            b_t = c_vec[0] + c_vec[1] * z_t + c_vec[2] * mu_t

            W_z_t = alpha_vec * mu_t

            pred = lam * b_t + (1.0 - lam) * W_z_t

            total_sq_err += np.sum((pred - target) ** 2)
            count += target.size

        return total_sq_err / count if count > 0 else np.inf

    # Initialize: alpha=0.5, c=[bias_const, weight_self, weight_neigh]
    init_params = np.concatenate([np.full(n_states, 0.5), [0.0, 1.0, 0.0]])


    res = minimize(objective, x0=init_params, method='L-BFGS-B')

    best_alpha = res.x[:n_states]
    best_c = res.x[n_states:]

    spectral_radius_max = 0.0
    all_eigenvalues = []
    eigenvalues_by_system = []

    for run_name, _, Abar in run_cache:
        W_eff = (1.0 - lam) * (best_alpha[:, None] * Abar)
        eigvals = np.linalg.eigvals(W_eff)
        eigvals_list = [complex(ev) for ev in np.asarray(eigvals).ravel()]
        spectral_radius_t = float(np.max(np.abs(eigvals)))

        spectral_radius_max = max(spectral_radius_max, spectral_radius_t)
        all_eigenvalues.extend(eigvals_list)
        eigenvalues_by_system.append(
            {
                'run': run_name,
                'eigenvalues': eigvals_list,
                'spectral_radius': spectral_radius_t,
            }
        )

    return {
        'lambda': lam,
        'alpha_vec': best_alpha,
        'c': best_c,
        'loss': float(res.fun),
        'spectral_radius': float(spectral_radius_max),
        'all_eigenvalues': np.asarray(all_eigenvalues, dtype=complex),
        'eigenvalues_by_system': eigenvalues_by_system,
    }

In [ ]:
def build_masked_W_from_beta(neighbors, beta):
    """Build row-stochastic W from per-agent learnable beta scores on valid neighbors."""
    beta = np.asarray(beta, dtype=float)
    n = beta.shape[0]
    W = np.zeros((n, n), dtype=float)
    for i in range(n):
        ns = neighbors.get(i, [i])
        if not ns:
            ns = [i]
        ns = [int(j) for j in ns if 0 <= int(j) < n]
        if not ns:
            ns = [i]
        row_scores = beta[i, ns]
        row_probs = softmax_stable(row_scores)
        W[i, ns] = row_probs
    return W


def rollout_predict_neighbor_weight_model(z0, *, neighbors, lam, beta, T, c):
    """Rollout with learned neighbor-specific weights and shared bias coefficients c0,c1,c2."""
    z0 = np.asarray(z0, dtype=float).ravel()
    T = int(T)
    W = build_masked_W_from_beta(neighbors, beta)
    out = np.zeros((T + 1, z0.shape[0]), dtype=float)
    out[0] = z0
    c = np.asarray(c, dtype=float).ravel()
    for t in range(T):
        mu = W @ out[t]
        b = c[0] + c[1] * out[t] + c[2] * mu
        out[t + 1] = float(lam) * b + (1.0 - float(lam)) * mu
    return out


def fit_neighbor_weight_model_at_lambda(run_records, lam):
    """Fit per-agent neighbor scores beta(i,j) over valid edges and shared bias c at fixed lambda."""
    lam = float(lam)
    n_states = run_records[0]['trajectory'].shape[1]

    cache = []
    for rec in run_records:
        x = np.asarray(rec['trajectory'], dtype=float)
        neighbors = rec['neighbors']
        cache.append((x, neighbors))

    def unpack(params):
        beta = params[: n_states * n_states].reshape(n_states, n_states)
        c = params[n_states * n_states :]
        return beta, c

    def objective(params):
        beta, c = unpack(params)
        total_sq_err = 0.0
        count = 0

        for x, neighbors in cache:
            W = build_masked_W_from_beta(neighbors, beta)
            z_t = x[:-1]
            target = x[1:]

            mu_t = z_t @ W.T
            b_t = c[0] + c[1] * z_t + c[2] * mu_t
            pred = lam * b_t + (1.0 - lam) * mu_t

            total_sq_err += float(np.sum((pred - target) ** 2))
            count += int(target.size)

        return total_sq_err / count if count > 0 else np.inf

    init_beta = np.zeros((n_states, n_states), dtype=float)
    init_c = np.array([0.0, 1.0, 0.0], dtype=float)
    init_params = np.concatenate([init_beta.ravel(), init_c])

    res = minimize(objective, x0=init_params, method='L-BFGS-B')
    best_beta, best_c = unpack(res.x)

    spectral_radius_max = 0.0
    for _, neighbors in cache:
        W = build_masked_W_from_beta(neighbors, best_beta)
        eigvals = np.linalg.eigvals((1.0 - lam) * W)
        spectral_radius_max = max(spectral_radius_max, float(np.max(np.abs(eigvals))))

    return {
        'lambda': lam,
        'beta': np.asarray(best_beta, dtype=float),
        'c': np.asarray(best_c, dtype=float),
        'loss': float(res.fun),
        'spectral_radius': float(spectral_radius_max),
    }

## Global Function And Per-Run Rollouts

- One shared function is learned from all trajectories (single learning signal).
- The function generates run- and state-dependent $W_t$ and $b_t$ at a given timestep
- There is no static global $W$ matrix and no post-hoc masking stage.

$$z^{(r)}(t+1)=\lambda\,b_t^{(r)}+(1-\lambda)\,W_t^{(r)}z^{(r)}(t)$$

- Horizon policy for each run $r$:

$$T_r^{\text{used}}=\min(H, T_r)$$

- Given the inital time slice, $W_t$ and $b_t$ are predicted and used to predict subsequent time slices

In [ ]:
def multi_step_validation_error_dynamic(run, neighbors, model, K=None):
    T = len(run) - 1
    if K is None:
        K = T
    K = int(min(K, T))
    if K <= 1:
        return np.nan
    pred, _, _ = rollout_predict_dynamic(run[0], neighbors=neighbors, model=model, T=K)
    return float(np.mean((pred[1:K] - run[1:K]) ** 2))


def run_diagnostics_table(run_name, run, neighbors, model, validation_horizon=None):
    rollout, _, _ = rollout_predict_dynamic(run[0], neighbors=neighbors, model=model, T=len(run) - 1)
    mse_rollout = float(np.mean((run[1:] - rollout[1:]) ** 2))
    mse_multi = multi_step_validation_error_dynamic(run, neighbors=neighbors, model=model, K=validation_horizon)

    return {
        'run': run_name,
        'lambda': float(model['lambda']),
        'loss_train_global': float(model['loss']),
        'mse_rollout_full': mse_rollout,
        'mse_multistep_mandatory': float(mse_multi),
        'spectral_radius_max': float(model['spectral_radius']),
    }


def plot_predicted_vs_observed(run_name, observed, predicted, agent_ids, horizon):
    T = min(int(horizon), observed.shape[0] - 1, predicted.shape[0] - 1)
    t = np.arange(T + 1)

    palette = np.array(
        list(plt.cm.tab20.colors) + list(plt.cm.Set3.colors) + list(plt.cm.Dark2.colors),
    )[: len(agent_ids)]

    plt.figure(figsize=(10.2, 5.6))
    for i, _ in enumerate(agent_ids):
        base_rgb = np.asarray(palette[i][:3], dtype=float)
        obs_rgb = 0.60 * np.array([0.72, 0.72, 0.72]) + 0.40 * base_rgb

        plt.plot(
            t,
            observed[: T + 1, i],
            color=(*obs_rgb, 0.42),
            linewidth=0.9,
            marker='o',
            markersize=1.9,
        )
        plt.plot(
            t,
            predicted[: T + 1, i],
            color=(*base_rgb, 0.92),
            linewidth=1.35,
        )

    legend_handles = [
        Line2D([0], [0], color=(0.45, 0.45, 0.45, 0.45), linewidth=1.0, marker='o', markersize=3, label='observed'),
        Line2D([0], [0], color=(0.20, 0.20, 0.20, 0.95), linewidth=1.6, label='predicted'),
    ]
    plt.title(f'{run_name}: predicted vs observed rollout (first {T} slices)')
    plt.xlabel('time slice')
    plt.ylabel('stance score')
    plt.legend(handles=legend_handles, loc='upper right', frameon=False)
    plt.tight_layout()
    plt.show()


def plot_mae_by_time(run_name, observed, predicted, horizon):
    T = min(int(horizon), observed.shape[0] - 1, predicted.shape[0] - 1)
    abs_err = np.abs(predicted[: T + 1] - observed[: T + 1])
    mae_t = abs_err.mean(axis=1)

    plt.figure(figsize=(7.6, 3.6))
    plt.plot(np.arange(T + 1), mae_t, color='tab:blue', linewidth=1.5)
    plt.title(f'{run_name}: MAE by time slice (first {T} slices)')
    plt.xlabel('time slice')
    plt.ylabel('MAE')
    plt.xlim(0, max(1, T))
    plt.tight_layout()
    plt.show()


def build_observed_trajectory_only(data, global_agent_ids):
    """Trajectory using *slice-0 self-post* when present; else NaN. (No predictor.)"""
    min_t = int(data.get('min_time_slice', 0))
    max_t = int(data['max_time_slice'])
    T = max_t - min_t
    n = len(global_agent_ids)
    agent_to_idx = {a: i for i, a in enumerate(global_agent_ids)}
    traj = np.full((T + 1, n), np.nan, dtype=float)

    # Only slice-0 observed is defined as: self-post at run-relative slice0 (min_t)
    first_self_posts = data.get('first_self_posts', {}) or {}
    for a in global_agent_ids:
        info = first_self_posts.get(a)
        if info is None:
            continue
        if int(info.get('time_slice', -10**9)) == min_t:
            traj[0, agent_to_idx[a]] = float(info['stance_score'])

    return traj


RUNS_SORTED = sorted(RUN_TRAJ.keys())
TRAINING_RUN_RECORDS = [
    {
        'run_name': rn,
        'trajectory': RUN_TRAJ[rn],
        'neighbors': RUN_NEIGHBORS[rn],
    }
    for rn in RUNS_SORTED
]


print(PARAMS['lambda_grid'])

GLOBAL_FUNCTION_MODEL = fit_global_function_model(
    run_records=TRAINING_RUN_RECORDS,
    lambda_grid=PARAMS['lambda_grid'],
)


print('Global dynamic function learned from stacked transitions across all runs.')
print('  lambda =', GLOBAL_FUNCTION_MODEL['lambda'])
print('  loss =', GLOBAL_FUNCTION_MODEL['loss'])
print('  spectral_radius_max =', GLOBAL_FUNCTION_MODEL['spectral_radius'])
print('  theta_w =')
print(np.array2string(np.asarray(GLOBAL_FUNCTION_MODEL['theta_w']), precision=6, separator=', ', threshold=np.inf, max_line_width=200))
print('  theta_b =')
print(np.array2string(np.asarray(GLOBAL_FUNCTION_MODEL['theta_b']), precision=6, separator=', ', threshold=np.inf, max_line_width=200))
print('  all_eigenvalues =')
print(np.array2string(np.asarray(GLOBAL_FUNCTION_MODEL['all_eigenvalues'], dtype=complex), precision=6, separator=', ', threshold=np.inf, max_line_width=200))

In [ ]:
def fit_adj_function_model(run_records, lambda_grid):
    """
    Iterate lambda grid, retain admissible (spectral radius < 1), and return best-loss model
    with lambda diagnostics table. Matches global-model packaging format.
    """
    per_lambda_models = []
    for lam in lambda_grid:
        m = fit_adj_function_at_lambda(run_records, lam)
        per_lambda_models.append(m)

    diag_rows = [
        {
            'lambda': m['lambda'],
            'loss': m['loss'],
            'spectral_radius': m['spectral_radius'],
        }
        for m in per_lambda_models
    ]
    lambda_diagnostics = pd.DataFrame(diag_rows).sort_values('lambda').reset_index(drop=True)

    admissible = [
        m for m in per_lambda_models
        if np.isfinite(m['spectral_radius']) and m['spectral_radius'] < 1.0
    ]
    if not admissible:
        raise RuntimeError('No admissible lambda found (all results unstable/non-finite).')

    best = min(admissible, key=lambda m: m['loss'])
    model = dict(best)
    model['lambda_diagnostics'] = lambda_diagnostics
    model['per_lambda_models'] = per_lambda_models
    return model


GLOBAL_FUNCTION_ADJ_MODEL = fit_adj_function_model(
    run_records=TRAINING_RUN_RECORDS,
    lambda_grid=PARAMS['lambda_grid'],
)

print('Global adjacency dynamic function learned from stacked transitions across all runs.')
print('  lambda =', GLOBAL_FUNCTION_ADJ_MODEL['lambda'])
print('  loss =', GLOBAL_FUNCTION_ADJ_MODEL['loss'])
print('  spectral_radius_max =', GLOBAL_FUNCTION_ADJ_MODEL['spectral_radius'])
print('  alpha_vec =')
print(np.array2string(np.asarray(GLOBAL_FUNCTION_ADJ_MODEL['alpha_vec']), precision=6, separator=', ', threshold=np.inf, max_line_width=200))
print('  c =')
print(np.array2string(np.asarray(GLOBAL_FUNCTION_ADJ_MODEL['c']), precision=6, separator=', ', threshold=np.inf, max_line_width=200))

adj_all_eigs = GLOBAL_FUNCTION_ADJ_MODEL.get('all_eigenvalues')
if adj_all_eigs is None:
    adj_all_eigs = []
    for rn in RUNS_SORTED:
        Abar = adjacency_to_row_stochastic(RUN_NEIGHBORS[rn], len(GLOBAL_FUNCTION_ADJ_MODEL['alpha_vec']))
        W_eff = (1.0 - float(GLOBAL_FUNCTION_ADJ_MODEL['lambda'])) * (np.asarray(GLOBAL_FUNCTION_ADJ_MODEL['alpha_vec'])[:, None] * Abar)
        adj_all_eigs.extend(np.linalg.eigvals(W_eff).tolist())

print('  all_eigenvalues =')
print(np.array2string(np.asarray(adj_all_eigs, dtype=complex), precision=6, separator=', ', threshold=np.inf, max_line_width=200))

In [ ]:
def fit_neighbor_weight_model(run_records, lambda_grid):
    """Lambda grid search for the neighbor-parameterized W model."""
    per_lambda_models = []
    for lam in lambda_grid:
        m = fit_neighbor_weight_model_at_lambda(run_records, lam)
        per_lambda_models.append(m)

    diag_rows = [
        {
            'lambda': m['lambda'],
            'loss': m['loss'],
            'spectral_radius': m['spectral_radius'],
        }
        for m in per_lambda_models
    ]
    lambda_diagnostics = pd.DataFrame(diag_rows).sort_values('lambda').reset_index(drop=True)

    admissible = [
        m for m in per_lambda_models
        if np.isfinite(m['spectral_radius']) and m['spectral_radius'] < 1.0
    ]
    if not admissible:
        raise RuntimeError('No admissible lambda found for neighbor-weight model.')

    best = min(admissible, key=lambda m: m['loss'])
    model = dict(best)
    model['lambda_diagnostics'] = lambda_diagnostics
    model['per_lambda_models'] = per_lambda_models
    return model


GLOBAL_FUNCTION_NEIGHBOR_MODEL = fit_neighbor_weight_model(
    run_records=TRAINING_RUN_RECORDS,
    lambda_grid=PARAMS['lambda_grid'],
)

print('Global neighbor-weight dynamic function learned from stacked transitions across all runs.')
print('  lambda =', GLOBAL_FUNCTION_NEIGHBOR_MODEL['lambda'])
print('  loss =', GLOBAL_FUNCTION_NEIGHBOR_MODEL['loss'])
print('  spectral_radius_max =', GLOBAL_FUNCTION_NEIGHBOR_MODEL['spectral_radius'])
print('  beta =')
print(np.array2string(np.asarray(GLOBAL_FUNCTION_NEIGHBOR_MODEL['beta']), precision=6, separator=', ', threshold=np.inf, max_line_width=200))
print('  c =')
print(np.array2string(np.asarray(GLOBAL_FUNCTION_NEIGHBOR_MODEL['c']), precision=6, separator=', ', threshold=np.inf, max_line_width=200))

neighbor_all_eigs = GLOBAL_FUNCTION_NEIGHBOR_MODEL.get('all_eigenvalues')
if neighbor_all_eigs is None:
    neighbor_all_eigs = []
    for rn in RUNS_SORTED:
        W = build_masked_W_from_beta(RUN_NEIGHBORS[rn], np.asarray(GLOBAL_FUNCTION_NEIGHBOR_MODEL['beta']))
        eigvals = np.linalg.eigvals((1.0 - float(GLOBAL_FUNCTION_NEIGHBOR_MODEL['lambda'])) * W)
        neighbor_all_eigs.extend(eigvals.tolist())

print('  all_eigenvalues =')
print(np.array2string(np.asarray(neighbor_all_eigs, dtype=complex), precision=6, separator=', ', threshold=np.inf, max_line_width=200))

In [ ]:
lambda_diag = GLOBAL_FUNCTION_MODEL['lambda_diagnostics']
plt.figure(figsize=(7.8, 3.6))
plt.plot(lambda_diag['lambda'], lambda_diag['loss'], marker='o', linewidth=1.5)
plt.xlabel('lambda')
plt.ylabel('loss (global MSE per coordinate)')
plt.title('Global lambda grid diagnostics: GLOBAL_FUNCTION_MODEL')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


lambda_diag_adj = GLOBAL_FUNCTION_ADJ_MODEL['lambda_diagnostics']
plt.figure(figsize=(7.8, 3.6))
plt.plot(lambda_diag_adj['lambda'], lambda_diag_adj['loss'], marker='o', linewidth=1.5)
plt.xlabel('lambda')
plt.ylabel('loss (global MSE per coordinate)')
plt.title('Global lambda grid diagnostics: GLOBAL_FUNCTION_ADJ_MODEL')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


lambda_diag_neighbor = GLOBAL_FUNCTION_NEIGHBOR_MODEL['lambda_diagnostics']
plt.figure(figsize=(7.8, 3.6))
plt.plot(lambda_diag_neighbor['lambda'], lambda_diag_neighbor['loss'], marker='o', linewidth=1.5)
plt.xlabel('lambda')
plt.ylabel('loss (global MSE per coordinate)')
plt.title('Global lambda grid diagnostics: GLOBAL_FUNCTION_NEIGHBOR_MODEL')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


def rollout_predict_for_model(z0, neighbors, model, T):
    """Dispatch rollout logic based on model parameterization."""
    if 'theta_w' in model and 'theta_b' in model:
        return rollout_predict_dynamic(z0, neighbors=neighbors, model=model, T=T)

    if 'beta' in model and 'c' in model:
        traj = rollout_predict_neighbor_weight_model(
            z0,
            neighbors=neighbors,
            lam=float(model['lambda']),
            beta=np.asarray(model['beta'], dtype=float),
            T=int(T),
            c=np.asarray(model['c'], dtype=float),
        )
        return traj, [], []

    if 'alpha_vec' in model and 'c' in model:
        z0 = np.asarray(z0, dtype=float).ravel()
        Abar = adjacency_to_row_stochastic(neighbors, z0.shape[0])
        traj = rollout_predict_agent_alpha_param_bias(
            z0,
            Abar=Abar,
            lam=float(model['lambda']),
            alpha_vec=np.asarray(model['alpha_vec'], dtype=float),
            T=int(T),
            c=np.asarray(model['c'], dtype=float),
        )
        # Match the output signature used elsewhere: (trajectory, Ws, bs).
        return traj, [], []

    raise KeyError("Unsupported model format: expected either (theta_w, theta_b), (alpha_vec, c), or (beta, c).")


def build_rollouts_and_diagnostics_for_model(model):
    run_diagnostics = []
    run_rollouts = {}
    run_horizons = {}

    horizon_cap = int(PARAMS['rollout_horizon_cap'])
    val_h = int(PARAMS['validation_horizon'])

    for rn in RUNS_SORTED:
        observed_full = RUN_TRAJ[rn]
        available_horizon = observed_full.shape[0] - 1
        effective_horizon = min(horizon_cap, available_horizon)
        run_horizons[rn] = int(effective_horizon)

        rollout, _, _ = rollout_predict_for_model(
            observed_full[0],
            neighbors=RUN_NEIGHBORS[rn],
            model=model,
            T=effective_horizon,
        )
        run_rollouts[rn] = rollout

        observed_used = observed_full[: effective_horizon + 1]
        if effective_horizon <= 0:
            diag = {
                'run': rn,
                'lambda': float(model['lambda']),
                'loss_train_global': float(model['loss']),
                'mse_rollout_full': np.nan,
                'mse_multistep_mandatory': np.nan,
                'spectral_radius_max': float(model['spectral_radius']),
                'rollout_horizon_used': int(effective_horizon),
            }
        else:
            mse_rollout = float(np.mean((observed_used[1:] - rollout[1:]) ** 2))

            K = int(min(val_h, effective_horizon))
            if K <= 1:
                mse_multi = np.nan
            else:
                pred_k, _, _ = rollout_predict_for_model(
                    observed_used[0],
                    neighbors=RUN_NEIGHBORS[rn],
                    model=model,
                    T=K,
                )
                mse_multi = float(np.mean((pred_k[1:K] - observed_used[1:K]) ** 2))

            diag = {
                'run': rn,
                'lambda': float(model['lambda']),
                'loss_train_global': float(model['loss']),
                'mse_rollout_full': mse_rollout,
                'mse_multistep_mandatory': float(mse_multi),
                'spectral_radius_max': float(model['spectral_radius']),
                'rollout_horizon_used': int(effective_horizon),
            }
        run_diagnostics.append(diag)

    run_diagnostics_df = pd.DataFrame(run_diagnostics).sort_values('run').reset_index(drop=True)
    return run_diagnostics, run_rollouts, run_horizons, run_diagnostics_df


RUN_DIAGNOSTICS, RUN_ROLLOUTS, RUN_HORIZONS, RUN_DIAGNOSTICS_DF = build_rollouts_and_diagnostics_for_model(
    GLOBAL_FUNCTION_MODEL
)

RUN_DIAGNOSTICS_ADJ, RUN_ROLLOUTS_ADJ, RUN_HORIZONS_ADJ, RUN_DIAGNOSTICS_ADJ_DF = build_rollouts_and_diagnostics_for_model(
    GLOBAL_FUNCTION_ADJ_MODEL
)

RUN_DIAGNOSTICS_NEIGHBOR, RUN_ROLLOUTS_NEIGHBOR, RUN_HORIZONS_NEIGHBOR, RUN_DIAGNOSTICS_NEIGHBOR_DF = build_rollouts_and_diagnostics_for_model(
    GLOBAL_FUNCTION_NEIGHBOR_MODEL
)

print('\nPer-run diagnostics computed for GLOBAL_FUNCTION_MODEL, GLOBAL_FUNCTION_ADJ_MODEL, and GLOBAL_FUNCTION_NEIGHBOR_MODEL.')
print('GLOBAL_FUNCTION_MODEL runs:', len(RUN_DIAGNOSTICS_DF), 'lambda:', GLOBAL_FUNCTION_MODEL['lambda'])
print('GLOBAL_FUNCTION_ADJ_MODEL runs:', len(RUN_DIAGNOSTICS_ADJ_DF), 'lambda:', GLOBAL_FUNCTION_ADJ_MODEL['lambda'])
print('GLOBAL_FUNCTION_NEIGHBOR_MODEL runs:', len(RUN_DIAGNOSTICS_NEIGHBOR_DF), 'lambda:', GLOBAL_FUNCTION_NEIGHBOR_MODEL['lambda'])

In [ ]:
# --- Utility metrics: predicted vs observed distributional alignment ---
def _wasserstein_1d_equal_weight(x, y):
    """1D Wasserstein distance with equal weights; no SciPy dependency."""
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if x.size == 0 or y.size == 0:
        return np.nan
    x = np.sort(x)
    y = np.sort(y)
    n = x.size
    m = y.size
    if n == m:
        return float(np.mean(np.abs(x - y)))
    q = np.linspace(0.0, 1.0, max(n, m), endpoint=True)
    xq = np.quantile(x, q)
    yq = np.quantile(y, q)
    return float(np.mean(np.abs(xq - yq)))


def _steady_state_slice_indices(T, window=5):
    T = int(T)
    window = int(max(1, window))
    if T <= 0:
        return np.array([], dtype=int)
    start = max(0, (T + 1) - window)
    return np.arange(start, T + 1, dtype=int)


def compare_predicted_observed_metrics(
    observed,
    predicted,
    *,
    horizon,
    steady_window=5,
    clip=(-1.0, 1.0),
    run_name=None,
    plot=True,
    title_prefix=None,
    ):
    """
    Compute and (optionally) plot alignment between predicted vs observed opinions.

    Metrics computed:
      - Per-time mean and variance across agents
      - Per-time 1D Wasserstein distance across agent distributions
      - Steady-state summaries over last `steady_window` slices:
          mean/var averaged over time, plus Wasserstein averaged over time.

    Inputs:
      observed, predicted: arrays shaped (T+1, N) on the same agent index.
      horizon: number of transitions to evaluate (caps to available length).
    """
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    T = min(int(horizon), observed.shape[0] - 1, predicted.shape[0] - 1)
    if T <= 0:
        return {
            'run': run_name,
            'horizon_used': int(T),
            'steady_window': int(steady_window),
            'steady_mean_observed': np.nan,
            'steady_mean_predicted': np.nan,
            'steady_var_observed': np.nan,
            'steady_var_predicted': np.nan,
            'steady_wasserstein': np.nan,
            'per_time': pd.DataFrame(),
        }

    lo, hi = clip
    obs = np.clip(observed[: T + 1], lo, hi)
    pred = np.clip(predicted[: T + 1], lo, hi)

    t = np.arange(T + 1)
    obs_mean = obs.mean(axis=1)
    pred_mean = pred.mean(axis=1)
    obs_var = obs.var(axis=1)
    pred_var = pred.var(axis=1)

    wdist = np.array([_wasserstein_1d_equal_weight(pred[k], obs[k]) for k in range(T + 1)], dtype=float)

    steady_idx = _steady_state_slice_indices(T, window=steady_window)
    steady_mean_obs = float(np.mean(obs_mean[steady_idx]))
    steady_mean_pred = float(np.mean(pred_mean[steady_idx]))
    steady_var_obs = float(np.mean(obs_var[steady_idx]))
    steady_var_pred = float(np.mean(pred_var[steady_idx]))
    steady_w = float(np.mean(wdist[steady_idx]))

    per_time = pd.DataFrame({
        't': t,
        'mean_observed': obs_mean,
        'mean_predicted': pred_mean,
        'var_observed': obs_var,
        'var_predicted': pred_var,
        'wasserstein_1d': wdist,
    })

    out = {
        'run': run_name,
        'horizon_used': int(T),
        'steady_window': int(steady_window),
        'steady_mean_observed': steady_mean_obs,
        'steady_mean_predicted': steady_mean_pred,
        'steady_var_observed': steady_var_obs,
        'steady_var_predicted': steady_var_pred,
        'steady_wasserstein': steady_w,
        'per_time': per_time,
    }

    if plot:
        if title_prefix is None:
            title_prefix = (run_name + ': ') if run_name else ''
        fig, axes = plt.subplots(3, 1, figsize=(7.8, 8.6), sharex=True)
        axes[0].plot(t, obs_mean, label='observed mean', color='tab:gray', linewidth=1.6)
        axes[0].plot(t, pred_mean, label='predicted mean', color='tab:blue', linewidth=1.6)
        axes[0].set_ylabel('mean')
        axes[0].legend(frameon=False, loc='best')
        axes[0].set_title(f"{title_prefix}distribution metrics (first {T} slices)")

        axes[1].plot(t, obs_var, label='observed var', color='tab:gray', linewidth=1.6)
        axes[1].plot(t, pred_var, label='predicted var', color='tab:blue', linewidth=1.6)
        axes[1].set_ylabel('variance')
        axes[1].legend(frameon=False, loc='best')

        axes[2].plot(t, wdist, label='Wasserstein-1D', color='tab:orange', linewidth=1.6)
        axes[2].set_ylabel('W1 distance')
        axes[2].set_xlabel('time slice')
        axes[2].legend(frameon=False, loc='best')
        axes[2].set_xlim(0, max(1, T))
        plt.tight_layout()
        plt.show()

        print('Steady-state window:', steady_idx[0], 'to', steady_idx[-1], f"(len={len(steady_idx)})")
        print('  steady_mean_observed =', steady_mean_obs)
        print('  steady_mean_predicted =', steady_mean_pred)
        print('  steady_var_observed =', steady_var_obs)
        print('  steady_var_predicted =', steady_var_pred)
        print('  steady_wasserstein =', steady_w)

    return out


def compare_predicted_observed_metrics_global(
    run_names,
    *,
    run_rollouts,
    run_horizons,
    horizon=20,
    steady_window=5,
    clip=(-1.0, 1.0),
    ):
    """Global aggregation across runs: averages per-time metrics across runs, then steady-state summaries."""
    per_time_rows = []
    used = []
    for rn in run_names:
        if rn not in RUN_TRAJ or rn not in run_rollouts:
            continue
        T = min(int(horizon), int(run_horizons.get(rn, 0)))
        if T <= 0:
            continue
        res = compare_predicted_observed_metrics(
            RUN_TRAJ[rn], run_rollouts[rn], horizon=T, steady_window=steady_window, clip=clip, run_name=rn, plot=False
        )
        pt = res['per_time'].copy()
        pt['run'] = rn
        per_time_rows.append(pt)
        used.append(rn)

    if not per_time_rows:
        return {
            'runs_used': [],
            'horizon': int(horizon),
            'steady_window': int(steady_window),
            'global_steady_mean_observed': np.nan,
            'global_steady_mean_predicted': np.nan,
            'global_steady_var_observed': np.nan,
            'global_steady_var_predicted': np.nan,
            'global_steady_wasserstein': np.nan,
            'per_time_mean': pd.DataFrame(),
        }

    all_pt = pd.concat(per_time_rows, ignore_index=True)
    per_time_mean = (
        all_pt.groupby('t')[['mean_observed', 'mean_predicted', 'var_observed', 'var_predicted', 'wasserstein_1d']]
        .mean()
        .reset_index()
        .sort_values('t')
    )
    T_eff = int(per_time_mean['t'].max())
    steady_idx = _steady_state_slice_indices(T_eff, window=steady_window)
    per_time_mean_idx = per_time_mean.set_index('t')
    global_steady_mean_obs = float(np.mean(per_time_mean_idx.loc[steady_idx, 'mean_observed']))
    global_steady_mean_pred = float(np.mean(per_time_mean_idx.loc[steady_idx, 'mean_predicted']))
    global_steady_var_obs = float(np.mean(per_time_mean_idx.loc[steady_idx, 'var_observed']))
    global_steady_var_pred = float(np.mean(per_time_mean_idx.loc[steady_idx, 'var_predicted']))
    global_steady_w = float(np.mean(per_time_mean_idx.loc[steady_idx, 'wasserstein_1d']))

    return {
        'runs_used': used,
        'horizon': int(horizon),
        'steady_window': int(steady_window),
        'global_steady_mean_observed': global_steady_mean_obs,
        'global_steady_mean_predicted': global_steady_mean_pred,
        'global_steady_var_observed': global_steady_var_obs,
        'global_steady_var_predicted': global_steady_var_pred,
        'global_steady_wasserstein': global_steady_w,
        'per_time_mean': per_time_mean,
    }


def compare_metrics_for_run(
    run_name,
    *,
    run_rollouts,
    run_horizons,
    horizon=20,
    steady_window=5,
    plot=True,
    title_prefix=None,
    ):
    obs = RUN_TRAJ[run_name]
    pred = run_rollouts[run_name]
    T = min(int(horizon), int(run_horizons.get(run_name, obs.shape[0] - 1)))
    return compare_predicted_observed_metrics(
        obs, pred, horizon=T, steady_window=steady_window, run_name=run_name, plot=plot, title_prefix=title_prefix
    )

In [ ]:
print('Plotting runs for GLOBAL_FUNCTION_MODEL:', RUNS_SORTED)

for rn in RUNS_SORTED:
    T = int(RUN_HORIZONS.get(rn, 0))
    if T <= 0:
        print(f"Skipping plots for {rn}: rollout horizon {T} (no post-slice steps).")
        continue

    obs = RUN_TRAJ[rn][: T + 1]
    pred = RUN_ROLLOUTS[rn]

    plot_predicted_vs_observed(rn, obs, pred, GLOBAL_AGENT_IDS, T)
    plot_mae_by_time(rn, obs, pred, T)

In [ ]:
print('Plotting runs for GLOBAL_FUNCTION_ADJ_MODEL:', RUNS_SORTED)

for rn in RUNS_SORTED:
    T = int(RUN_HORIZONS_ADJ.get(rn, 0))
    if T <= 0:
        print(f"Skipping plots for {rn}: rollout horizon {T} (no post-slice steps).")
        continue

    obs = RUN_TRAJ[rn][: T + 1]
    pred = RUN_ROLLOUTS_ADJ[rn]

    plot_predicted_vs_observed(rn, obs, pred, GLOBAL_AGENT_IDS, T)
    plot_mae_by_time(rn, obs, pred, T)


In [ ]:
print('Plotting runs for GLOBAL_FUNCTION_NEIGHBOR_MODEL:', RUNS_SORTED)

for rn in RUNS_SORTED:
    T = int(RUN_HORIZONS_NEIGHBOR.get(rn, 0))
    if T <= 0:
        print(f"Skipping plots for {rn}: rollout horizon {T} (no post-slice steps).")
        continue

    obs = RUN_TRAJ[rn][: T + 1]
    pred = RUN_ROLLOUTS_NEIGHBOR[rn]

    plot_predicted_vs_observed(rn, obs, pred, GLOBAL_AGENT_IDS, T)
    plot_mae_by_time(rn, obs, pred, T)

In [ ]:
# --- Distributional metrics on longer runs (horizon ~20): GLOBAL_FUNCTION_MODEL ---
candidate_runs = [rn for rn in RUNS_SORTED if int(RUN_HORIZONS.get(rn, 0)) >= 18]
runs_for_metrics = candidate_runs[:3]
print('Runs with horizon>=18:', len(candidate_runs))
print('Plotting distribution metrics for GLOBAL_FUNCTION_MODEL:', runs_for_metrics)

METRICS_BY_RUN = []
for rn in runs_for_metrics:
    res = compare_metrics_for_run(
        rn,
        run_rollouts=RUN_ROLLOUTS,
        run_horizons=RUN_HORIZONS,
        horizon=20,
        steady_window=5,
        plot=True,
        title_prefix='GLOBAL_FUNCTION_MODEL | ',
    )
    METRICS_BY_RUN.append({k: v for k, v in res.items() if k != 'per_time'})

METRICS_DF = pd.DataFrame(METRICS_BY_RUN)
print('Computed per-run distribution metrics for GLOBAL_FUNCTION_MODEL:', len(METRICS_DF))

In [ ]:
# --- Distributional metrics on longer runs (horizon ~20): GLOBAL_FUNCTION_ADJ_MODEL ---
candidate_runs_adj = [rn for rn in RUNS_SORTED if int(RUN_HORIZONS_ADJ.get(rn, 0)) >= 18]
runs_for_metrics_adj = candidate_runs_adj[:3]
print('Runs with horizon>=18:', len(candidate_runs_adj))
print('Plotting distribution metrics for GLOBAL_FUNCTION_ADJ_MODEL:', runs_for_metrics_adj)

METRICS_BY_RUN_ADJ = []
for rn in runs_for_metrics_adj:
    res = compare_metrics_for_run(
        rn,
        run_rollouts=RUN_ROLLOUTS_ADJ,
        run_horizons=RUN_HORIZONS_ADJ,
        horizon=20,
        steady_window=5,
        plot=True,
        title_prefix='GLOBAL_FUNCTION_ADJ_MODEL | ',
    )
    METRICS_BY_RUN_ADJ.append({k: v for k, v in res.items() if k != 'per_time'})

METRICS_DF_ADJ = pd.DataFrame(METRICS_BY_RUN_ADJ)
print('Computed per-run distribution metrics for GLOBAL_FUNCTION_ADJ_MODEL:', len(METRICS_DF_ADJ))

In [ ]:
# --- Distributional metrics on longer runs (horizon ~20): GLOBAL_FUNCTION_NEIGHBOR_MODEL ---
candidate_runs_neighbor = [rn for rn in RUNS_SORTED if int(RUN_HORIZONS_NEIGHBOR.get(rn, 0)) >= 18]
runs_for_metrics_neighbor = candidate_runs_neighbor[:3]
print('Runs with horizon>=18:', len(candidate_runs_neighbor))
print('Plotting distribution metrics for GLOBAL_FUNCTION_NEIGHBOR_MODEL:', runs_for_metrics_neighbor)

METRICS_BY_RUN_NEIGHBOR = []
for rn in runs_for_metrics_neighbor:
    res = compare_metrics_for_run(
        rn,
        run_rollouts=RUN_ROLLOUTS_NEIGHBOR,
        run_horizons=RUN_HORIZONS_NEIGHBOR,
        horizon=20,
        steady_window=5,
        plot=True,
        title_prefix='GLOBAL_FUNCTION_NEIGHBOR_MODEL | ',
    )
    METRICS_BY_RUN_NEIGHBOR.append({k: v for k, v in res.items() if k != 'per_time'})

METRICS_DF_NEIGHBOR = pd.DataFrame(METRICS_BY_RUN_NEIGHBOR)
print('Computed per-run distribution metrics for GLOBAL_FUNCTION_NEIGHBOR_MODEL:', len(METRICS_DF_NEIGHBOR))

In [ ]:
# --- Global distributional metrics (no plot): GLOBAL_FUNCTION_MODEL ---
global_runs = [rn for rn in RUNS_SORTED if int(RUN_HORIZONS.get(rn, 0)) >= 18]
GLOBAL_METRICS_FUNCTION = compare_predicted_observed_metrics_global(
    global_runs,
    run_rollouts=RUN_ROLLOUTS,
    run_horizons=RUN_HORIZONS,
    horizon=20,
    steady_window=5,
    clip=PARAMS.get('stance_clip', (-1.0, 1.0)),
 )

print('Global runs used (GLOBAL_FUNCTION_MODEL):', len(GLOBAL_METRICS_FUNCTION['runs_used']))
print('Global steady-state (avg over runs, last window): GLOBAL_FUNCTION_MODEL')
print('  global_steady_mean_observed =', GLOBAL_METRICS_FUNCTION['global_steady_mean_observed'])
print('  global_steady_mean_predicted =', GLOBAL_METRICS_FUNCTION['global_steady_mean_predicted'])
print('  global_steady_var_observed =', GLOBAL_METRICS_FUNCTION['global_steady_var_observed'])
print('  global_steady_var_predicted =', GLOBAL_METRICS_FUNCTION['global_steady_var_predicted'])
print('  global_steady_wasserstein =', GLOBAL_METRICS_FUNCTION['global_steady_wasserstein'])

In [ ]:
# --- Global distributional metrics (no plot): GLOBAL_FUNCTION_ADJ_MODEL ---
global_runs_adj = [rn for rn in RUNS_SORTED if int(RUN_HORIZONS_ADJ.get(rn, 0)) >= 18]
GLOBAL_METRICS_ADJ = compare_predicted_observed_metrics_global(
    global_runs_adj,
    run_rollouts=RUN_ROLLOUTS_ADJ,
    run_horizons=RUN_HORIZONS_ADJ,
    horizon=20,
    steady_window=5,
    clip=PARAMS.get('stance_clip', (-1.0, 1.0)),
 )

print('Global runs used (GLOBAL_FUNCTION_ADJ_MODEL):', len(GLOBAL_METRICS_ADJ['runs_used']))
print('Global steady-state (avg over runs, last window): GLOBAL_FUNCTION_ADJ_MODEL')
print('  global_steady_mean_observed =', GLOBAL_METRICS_ADJ['global_steady_mean_observed'])
print('  global_steady_mean_predicted =', GLOBAL_METRICS_ADJ['global_steady_mean_predicted'])
print('  global_steady_var_observed =', GLOBAL_METRICS_ADJ['global_steady_var_observed'])
print('  global_steady_var_predicted =', GLOBAL_METRICS_ADJ['global_steady_var_predicted'])
print('  global_steady_wasserstein =', GLOBAL_METRICS_ADJ['global_steady_wasserstein'])

In [ ]:
# --- Global distributional metrics (no plot): GLOBAL_FUNCTION_NEIGHBOR_MODEL ---
global_runs_neighbor = [rn for rn in RUNS_SORTED if int(RUN_HORIZONS_NEIGHBOR.get(rn, 0)) >= 18]
GLOBAL_METRICS_NEIGHBOR = compare_predicted_observed_metrics_global(
    global_runs_neighbor,
    run_rollouts=RUN_ROLLOUTS_NEIGHBOR,
    run_horizons=RUN_HORIZONS_NEIGHBOR,
    horizon=20,
    steady_window=5,
    clip=PARAMS.get('stance_clip', (-1.0, 1.0)),
 )

print('Global runs used (GLOBAL_FUNCTION_NEIGHBOR_MODEL):', len(GLOBAL_METRICS_NEIGHBOR['runs_used']))
print('Global steady-state (avg over runs, last window): GLOBAL_FUNCTION_NEIGHBOR_MODEL')
print('  global_steady_mean_observed =', GLOBAL_METRICS_NEIGHBOR['global_steady_mean_observed'])
print('  global_steady_mean_predicted =', GLOBAL_METRICS_NEIGHBOR['global_steady_mean_predicted'])
print('  global_steady_var_observed =', GLOBAL_METRICS_NEIGHBOR['global_steady_var_observed'])
print('  global_steady_var_predicted =', GLOBAL_METRICS_NEIGHBOR['global_steady_var_predicted'])
print('  global_steady_wasserstein =', GLOBAL_METRICS_NEIGHBOR['global_steady_wasserstein'])